### Data Ingestion



In [24]:
### Document Structure

from langchain_core.documents import Document

In [25]:
doc = Document(
    page_content="This is a sample document for testing purposes.",
    metadata={
        "source":"example.txt",
        "pages":1,
        "author": "Aneesh",
        "date_created": "2026-05-04"
    }
)
doc

Document(metadata={'source': 'example.txt', 'pages': 1, 'author': 'Aneesh', 'date_created': '2026-05-04'}, page_content='This is a sample document for testing purposes.')

In [26]:
## create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [27]:
import os

data_dir = "../data/text_files"
os.makedirs(data_dir, exist_ok=True)

sample_texts = {
    "python_intro.txt": """Python Programming Introduction

Python is a high-level, interpreted programming language known for its simplicity and readability.
It supports multiple programming paradigms, including procedural, object-oriented, and functional programming.

Python is widely used in web development, data analysis, artificial intelligence, scientific computing, and automation.
Its large standard library and ecosystem of third-party packages make it a versatile choice for developers.
""",
    "machine_learning_intro.txt": """Machine Learning Introduction

Machine learning is a branch of artificial intelligence that enables systems to learn patterns from data and improve performance over time.
Instead of explicitly programming every rule, developers train models that generalize from examples to make predictions or decisions.

Common learning styles include supervised learning, unsupervised learning, and reinforcement learning.
Machine learning is used in recommendation systems, fraud detection, forecasting, computer vision, and natural language processing.
"""
}

for filename, content in sample_texts.items():
    file_path = os.path.join(data_dir, filename)
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

Sample text files created successfully.


In [28]:
### TextLoader
from langchain_community.document_loaders import TextLoader

loader = TextLoader("data/text_files/python_intro.txt", encoding="utf-8")
document = loader.load()
print(document)

[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nIt supports multiple programming paradigms, including procedural, object-oriented, and functional programming.\n\nPython is widely used in web development, data analysis, artificial intelligence, scientific computing, and automation.\nIts large standard library and ecosystem of third-party packages make it a versatile choice for developers.\n')]


In [29]:
### DirectoryLoader
from langchain_community.document_loaders import DirectoryLoader

dir_Loader = DirectoryLoader(
    "data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding': "utf-8"},
    show_progress=False
)
documents = dir_Loader.load()
print(documents)

[Document(metadata={'source': 'data/text_files/machine_learning_intro.txt'}, page_content='Machine Learning Introduction\n\nMachine learning is a branch of artificial intelligence that enables systems to learn patterns from data and improve performance over time.\nInstead of explicitly programming every rule, developers train models that generalize from examples to make predictions or decisions.\n\nCommon learning styles include supervised learning, unsupervised learning, and reinforcement learning.\nMachine learning is used in recommendation systems, fraud detection, forecasting, computer vision, and natural language processing.\n'), Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='Python Programming Introduction\n\nPython is a high-level, interpreted programming language known for its simplicity and readability.\nIt supports multiple programming paradigms, including procedural, object-oriented, and functional programming.\n\nPython is widely used in web

In [30]:
### Text Chunking
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=220,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = splitter.split_documents(documents)
print(f"Loaded docs: {len(documents)}")
print(f"Created chunks: {len(chunks)}")
print("Sample chunk:\n", chunks[0].page_content[:220])

Loaded docs: 2
Created chunks: 9
Sample chunk:
 Machine Learning Introduction


In [31]:
### Vector Store + Retriever
from langchain_core.embeddings import FakeEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore

embeddings = FakeEmbeddings(size=128)
vector_store = InMemoryVectorStore(embedding=embeddings)
_ = vector_store.add_documents(chunks)

retriever = vector_store.as_retriever(search_kwargs={"k": 2})
print("Retriever is ready.")

Retriever is ready.


In [32]:
### Embedding Numericals (inspect)
sample_query = "What is machine learning?"
query_vector = embeddings.embed_query(sample_query)

print("Query vector length:", len(query_vector))
print("First 10 query values:", query_vector[:10])

chunk_vectors = embeddings.embed_documents([chunks[0].page_content])
print("\nChunk vector length:", len(chunk_vectors[0]))
print("First 10 chunk values:", chunk_vectors[0][:10])

Query vector length: 128
First 10 query values: [np.float64(-0.8892264170454833), np.float64(-1.1661019744860806), np.float64(1.5824758520168536), np.float64(-0.2048157139378626), np.float64(-1.1494402485745805), np.float64(1.0325552395522408), np.float64(0.4291025645335066), np.float64(-1.2360513332235563), np.float64(-1.746388531130803), np.float64(-0.30479683699796156)]

Chunk vector length: 128
First 10 chunk values: [np.float64(0.4840314880851629), np.float64(-1.0332975518372456), np.float64(-0.7580227875394525), np.float64(-0.04859774225910334), np.float64(2.1534393667294895), np.float64(-0.26922232236882476), np.float64(-0.35941733419431143), np.float64(-1.4016704813547651), np.float64(0.45724981465604914), np.float64(-0.9135409664435151)]


In [33]:
### Retrieval Test
query = "What is machine learning used for?"
results = retriever.invoke(query)

print("Query:", query)
for i, d in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Source:", d.metadata.get("source", "unknown"))
    print(d.page_content[:300])

Query: What is machine learning used for?

Result 1
Source: data/text_files/machine_learning_intro.txt
Machine Learning Introduction

Result 2
Source: data/text_files/python_intro.txt
Python is a high-level, interpreted programming language known for its simplicity and readability.
It supports multiple programming paradigms, including procedural, object-oriented, and functional programming.


In [34]:
### Install Gemini package (run once)
%pip install -q langchain-google-genai

/home/aneeshpuranik/agentic RAG/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
### Gemini API key setup
import os

# Set GOOGLE_API_KEY in your environment before running this cell.
GEMINI_API_KEY = ""

api_key = (GEMINI_API_KEY or os.getenv("GOOGLE_API_KEY", "")).strip()

if api_key:
    os.environ["GOOGLE_API_KEY"] = api_key
    if api_key.startswith("AIza"):
        print("Gemini API key loaded.")
    else:
        print("API key loaded, but format looks unusual. Gemini keys usually start with 'AIza'.")
else:
    print("Set GOOGLE_API_KEY in your environment, then run again.")

Gemini API key loaded.


In [49]:
### Final RAG answer with Gemini
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

active_retriever = retriever_real if "retriever_real" in globals() else retriever
question = "Explain python programming and machine learning in 10 words."

retrieved_docs = active_retriever.invoke(question)
context = "\n\n".join([doc.page_content for doc in retrieved_docs])

if not os.environ.get("GOOGLE_API_KEY"):
    print("Gemini API key not found. Add it in the previous cell and run again.")
else:
    prompt = ChatPromptTemplate.from_template(
        """You are a helpful assistant. Answer only from the provided context.

Context:
{context}

Question: {question}
"""
    )
    model_candidates = [
        "gemini-2.0-flash",
        "gemini-2.5-flash",
        "gemini-1.5-flash",
        "gemini-1.5-pro",
    ]

    last_error = None
    for model_name in model_candidates:
        try:
            llm = ChatGoogleGenerativeAI(model=model_name, temperature=0.2)
            chain = prompt | llm | StrOutputParser()
            answer = chain.invoke({"context": context, "question": question})
            print(f"Model used: {model_name}\n")
            print(answer)
            last_error = None
            break
        except Exception as e:
            last_error = e

    if last_error is not None:
        print("No listed model was available for this API key/project.")
        print("Last error:", last_error)

Model used: gemini-2.5-flash

Python is used in artificial intelligence; Machine Learning is introduced here.
